<a href="https://colab.research.google.com/github/CodeRichen/presidio-research/blob/master/data_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from presidio_evaluator import InputSample

train_samples = InputSample.read_dataset_json("../data/train_May-04-2026.json")
test_samples = InputSample.read_dataset_json("../data/test_May-04-2026.json")

print(f"train: {len(train_samples)} 筆")
print(f"test: {len(test_samples)} 筆")

# 檢查前幾筆有沒有正常解析
for s in train_samples[:2]:
    print(s.full_text)
    print(s.tags)
    print("---")

tokenizing input: 100%|██████████| 292/292 [00:01<00:00, 165.76it/s]

train: 1071 筆
test: 292 筆
Who's coming to Germany with me?
['O', 'O', 'O', 'O', 'GPE', 'O', 'O', 'O']
---
The name in the account is not correct, please change it to Ivan King
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'PERSON', 'PERSON']
---


In [ ]:
from spacy.tokens import DocBin
db = DocBin().from_disk("train_en.spacy")
docs = list(db.get_docs(nlp.vocab))

# 看前3筆
for doc in docs[:3]:
    print(f"文字: {doc.text}")
    print(f"ents: {list(doc.ents)}")
    print(f"tokens: {[t.text for t in doc]}")
    print("---")
for s in train_samples[:3]:
    print(f"full_text: {s.full_text}")
    print(f"tokens: {[t.text for t in s.tokens] if s.tokens else 'None'}")
    print(f"tags: {s.tags}")
    print("---")

文字: Who's coming to Germany with me?
ents: []
tokens: ['Who', "'s", 'coming', 'to', 'Germany', 'with', 'me', '?']
---
文字: The name in the account is not correct, please change it to Ivan King
ents: []
tokens: ['The', 'name', 'in', 'the', 'account', 'is', 'not', 'correct', ',', 'please', 'change', 'it', 'to', 'Ivan', 'King']
---
文字: Have you been to a Harvey Leach concert before?
ents: []
tokens: ['Have', 'you', 'been', 'to', 'a', 'Harvey', 'Leach', 'concert', 'before', '?']
---
full_text: Who's coming to Germany with me?
tokens: ['Who', "'s", 'coming', 'to', 'Germany', 'with', 'me', '?']
tags: ['O', 'O', 'O', 'O', 'GPE', 'O', 'O', 'O']
---
full_text: The name in the account is not correct, please change it to Ivan King
tokens: ['The', 'name', 'in', 'the', 'account', 'is', 'not', 'correct', ',', 'please', 'change', 'it', 'to', 'Ivan', 'King']
tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'PERSON', 'PERSON']
---
full_text: Have you been to a Harvey Leach concert

In [ ]:
import spacy
from spacy.tokens import DocBin, Doc

nlp = spacy.blank("en")
def samples_to_docbin_from_tags(samples, output_path):
    db = DocBin()
    skipped = 0

    for sample in samples:
        if not sample.tokens or not sample.tags:
            skipped += 1
            continue

        words = [t.text for t in sample.tokens]
        spaces = [t.whitespace_ != "" for t in sample.tokens]
        doc = Doc(nlp.vocab, words=words, spaces=spaces)

        ents = []
        start = None
        current_label = None

        for i, tag in enumerate(sample.tags):
            # 判斷是 BIO 格式還是純標籤格式
            if tag.startswith("B-"):
                # BIO 格式：B- 開頭
                if start is not None:
                    ents.append(spacy.tokens.Span(doc, start, i, label=current_label))
                start = i
                current_label = tag[2:]
            elif tag.startswith("I-"):
                pass  # 繼續同一實體
            elif tag == "O":
                # 實體結束
                if start is not None:
                    ents.append(spacy.tokens.Span(doc, start, i, label=current_label))
                    start = None
                    current_label = None
            else:
                # 純標籤格式（如 "PERSON", "GPE"）
                if tag == current_label:
                    pass  # 同一實體繼續
                else:
                    # 前一個實體結束，新實體開始
                    if start is not None:
                        ents.append(spacy.tokens.Span(doc, start, i, label=current_label))
                    start = i
                    current_label = tag

        # 處理最後一個實體
        if start is not None:
            ents.append(spacy.tokens.Span(doc, start, len(doc), label=current_label))

        doc.ents = spacy.util.filter_spans(ents)
        db.add(doc)

    db.to_disk(output_path)
    print(f"✅ 儲存 {len(samples) - skipped} 筆 → {output_path}（跳過 {skipped} 筆）")

# 重新轉換
samples_to_docbin_from_tags(train_samples, "train_en.spacy")
samples_to_docbin_from_tags(test_samples, "dev_en.spacy")



✅ 儲存 1071 筆 → train_en.spacy（跳過 0 筆）
✅ 儲存 292 筆 → dev_en.spacy（跳過 0 筆）
✅ 儲存 1071 筆 → train_en.spacy（跳過 0 筆）
✅ 儲存 292 筆 → dev_en.spacy（跳過 0 筆）


In [ ]:
# InputSample.create_spacy_dataset(train_samples, output_path="train_en.spacy")
# InputSample.create_spacy_dataset(test_samples, output_path="dev_en.spacy")

In [ ]:
from spacy.tokens import DocBin
import spacy
from collections import Counter

nlp = spacy.blank("en")

# 讀回來檢查
for path, name in [("train_en.spacy", "Train"), ("dev_en.spacy", "Dev")]:
    db = DocBin().from_disk(path)
    docs = list(db.get_docs(nlp.vocab))

    label_counter = Counter()
    for doc in docs:
        for ent in doc.ents:
            label_counter[ent.label_] += 1

    print(f"=== {name} ===")
    print(f"總筆數: {len(docs)}")
    print(f"實體分布: {dict(label_counter)}")
    print()

=== Train ===
總筆數: 1071
實體分布: {'GPE': 306, 'PERSON': 606, 'DATE_TIME': 88, 'PHONE_NUMBER': 90, 'IP_ADDRESS': 14, 'EMAIL_ADDRESS': 49, 'STREET_ADDRESS': 373, 'ORGANIZATION': 205, 'CREDIT_CARD': 85, 'DOMAIN_NAME': 37, 'TITLE': 56, 'AGE': 40, 'US_DRIVER_LICENSE': 5, 'NRP': 30, 'US_SSN': 16, 'ZIP_CODE': 19, 'IBAN_CODE': 11}

=== Dev ===
總筆數: 292
實體分布: {'PERSON': 183, 'CREDIT_CARD': 32, 'STREET_ADDRESS': 87, 'DATE_TIME': 21, 'ORGANIZATION': 15, 'AGE': 26, 'ZIP_CODE': 14, 'TITLE': 28, 'NRP': 25, 'GPE': 63, 'PHONE_NUMBER': 2}



In [ ]:
!ls /content/drive/MyDrive/PIImodel/

config.cfg     config_en.cfg.gdoc  dev_en.spacy      output
config_en.cfg  data-test.ipynb	   first_test.ipynb  train_en.spacy


In [1]:
from google.colab import drive
# !python -m spacy download zh_core_web_lg
!python -m spacy download en_core_web_lg
# !python -m spacy init config config.cfg \
#     --lang en \
#     --pipeline ner \
#     --optimize efficiency \
#     --gpu --force \
drive.mount('/content/drive')
# !python -m spacy train config.cfg \
!python -m spacy train /content/drive/MyDrive/PIImodel/config_en.cfg \
    --output /content/drive/MyDrive/PIImodel/output \
    --paths.train /content/drive/MyDrive/PIImodel/train_en.spacy \
    --paths.dev /content/drive/MyDrive/PIImodel/dev_en.spacy \
    --gpu-id 0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Mounted at /content/drive
✔ Created output directory: /content/drive/MyDrive/PIImodel/output
ℹ Saving to output directory:
/content/drive/MyDrive/PIImodel/output
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!zip -r /content/drive/MyDrive/PIImodel/model-best.zip /content/drive/MyDrive/PIImodel/output/model-best

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
updating: content/drive/MyDrive/PIImodel/output/model-best/ (stored 0%)
updating: content/drive/MyDrive/PIImodel/output/model-best/tok2vec/ (stored 0%)
updating: content/drive/MyDrive/PIImodel/output/model-best/tok2vec/cfg (stored 0%)
updating: content/drive/MyDrive/PIImodel/output/model-best/ner/ (stored 0%)
updating: content/drive/MyDrive/PIImodel/output/model-best/ner/model (deflated 8%)
updating: content/drive/MyDrive/PIImodel/output/model-best/ner/moves (deflated 59%)
updating: content/drive/MyDrive/PIImodel/output/model-best/ner/cfg (deflated 33%)
updating: content/drive/MyDrive/PIImodel/output/model-best/tokenizer (deflated 81%)
updating: content/drive/MyDrive/PIImodel/output/model-best/meta.json (deflated 63%)
updating: content/drive/MyDrive/PIImodel/output/model-best/config.cfg (deflated 61%)
updating: content/drive/MyDrive/PIImodel/output/model-best

In [ ]:
cd presidio-research\app
python -m spacy init config config.cfg --lang en --pipeline ner
python -m spacy train config.cfg  --output ./output   --paths.train train_en.spacy  --paths.dev dev_en.spacy

In [ ]:
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     63.09    0.00    0.00    0.00    0.00
  1     200        141.13   3291.16   61.79   64.75   59.09    0.62
  2     400        226.63   1821.27   59.86   59.76   59.97    0.60
  3     600         94.49   1207.25   62.50   64.85   60.31    0.62
  5     800         74.52    670.76   68.62   67.51   69.76    0.69
  8    1000        147.41    432.50   64.26   60.62   68.36    0.64
 11    1200         99.59    303.03   68.90   69.64   68.18    0.69
 15    1400         76.89    213.22   67.26   67.91   66.61    0.67
 19    1600        177.55    205.82   66.55   67.94   65.21    0.67
 25    1800        180.60    197.97   69.55   70.23   68.88    0.70
 32    2000        158.00    152.61   69.68   72.01   67.48    0.70
 40    2200         20.32     21.97   69.75   66.14   73.78    0.70
 50    2400         29.57     26.77   74.05   73.29   74.83    0.74
 61    2600          6.65      4.22   67.70   66.72   68.71    0.68
 71    2800        349.11    200.65   69.11   72.64   65.91    0.69
 82    3000        351.62    270.59   68.47   66.28   70.80    0.68
 92    3200        351.97    352.95   67.26   68.29   66.26    0.67
103    3400         56.99     37.67   70.69   73.00   68.53    0.71
113    3600         59.90     32.70   71.16   73.20   69.23    0.71
124    3800        222.65    123.71   70.01   73.14   67.13    0.70
134    4000       1161.94    264.96   67.59   67.01   68.18    0.68

In [ ]:
import spacy

nlp = spacy.load("./output/model-best")

print(nlp.get_pipe("ner").labels)

In [ ]:
from spacy.tokens import DocBin
db = DocBin().from_disk("train_en.spacy")
docs = list(db.get_docs(nlp.vocab))

print(f"共 {len(docs)} 筆")
for doc in docs[:3]:
    print(doc.text)
    print([(ent.text, ent.label_) for ent in doc.ents])
    print("---")

In [ ]:
import spacy

nlp = spacy.load("./output/model-best")

text = "Justin sent his email test@example.com to Kat in 6/8/2023 and she replied to him on 6/9/2023. They also discussed the new project at 10:00 AM."
doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

In [ ]:
import time
from datetime import datetime

import pyperclip
import spacy

# -----------------------------
# Load custom spaCy model
# -----------------------------
MODEL_PATH = "./output/model-best"

print("=" * 60)
print("Loading custom model...")
nlp = spacy.load(MODEL_PATH)
print("Model loaded successfully.")
print("NER Labels:", nlp.get_pipe("ner").labels)
print("=" * 60)

last_text = ""


def anonymize(text, entities):
    """
    Replace entities with their labels.
    """
    result = text

    # Replace from back to front to avoid index shifting
    for ent in sorted(entities, key=lambda x: x.start_char, reverse=True):
        replacement = f"<{ent.label_}>"
        result = (
            result[:ent.start_char]
            + replacement
            + result[ent.end_char:]
        )

    return result


print("Clipboard monitor started...")
print("Press Ctrl+C to stop.\n")

while True:

    try:
        current = pyperclip.paste()

        if current != last_text and current.strip():

            last_text = current

            print("\n" + "=" * 70)
            print("Clipboard Updated")
            print("=" * 70)
            print("Time :", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            print()

            print("Original Text")
            print("-" * 70)
            print(current)
            print()

            start = time.perf_counter()

            doc = nlp(current)

            elapsed = (time.perf_counter() - start) * 1000

            print("Detected Entities")
            print("-" * 70)

            if len(doc.ents) == 0:
                print("No entities detected.")

            else:

                statistics = {}

                for i, ent in enumerate(doc.ents, start=1):

                    statistics[ent.label_] = statistics.get(ent.label_, 0) + 1

                    print(f"Entity #{i}")
                    print(f"Text       : {ent.text}")
                    print(f"Label      : {ent.label_}")
                    print(f"Start      : {ent.start_char}")
                    print(f"End        : {ent.end_char}")
                    print()

                print("-" * 70)
                print("Statistics")

                for label, count in statistics.items():
                    print(f"{label:10} : {count}")

            print()

            anonymized = anonymize(current, doc.ents)

            print("-" * 70)
            print("Anonymized Text")
            print("-" * 70)
            print(anonymized)
            print()

            pyperclip.copy(anonymized)

            print("Clipboard updated successfully.")
            print(f"NER Processing Time : {elapsed:.2f} ms")
            print("=" * 70)

        time.sleep(0.3)

    except KeyboardInterrupt:
        print("\nProgram stopped.")
        break

    except Exception as e:
        print(e)
        time.sleep(1)